# WILDFIRE PREDICT - FEATURE EXTRACTOR

This module is responsible for loading the downloaded Sentinel-2 images, and running ResNet-18 as feature extractor. The module is created as a Jupyter Notebook to have the option of running the script in `GoogleColab`, if further GPU is required for the processing of the data.

## Libraries

In [1]:
# Libraries
import os
import numpy as np
import torch
import torch.nn as nn


# from google.colab import drive
from scripts.set_parameters import PARAMETERS
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (resnet18, ResNet18_Weights)
from torchvision.models.feature_extraction import create_feature_extractor

## Data Load

### Class `SentinelDataset` 

Create Class to encapsulate and easily manage the Sentinel data

In [2]:
class SentinelData(Dataset):
    def __init__(self, npz_file):
        data      = np.load(npz_file)
        self.x    = data['x']
        self.y    = data['y']
        self.keys = data['composite_key']

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        # Change/permute image from HeightWidthChannel to CHW as the CNN model expects
        pixel_data = torch.from_numpy(self.x[idx]).permute(2,0,1)
        fire_label = self.y[idx]
        composite_key = str(self.keys[idx])

        return {"pixel_data": pixel_data,
                "fire_label": fire_label,
                "composite_key": composite_key}
    
    def sample_summary(self, idx=0):

        sample = self[idx]

        print("SentinelDataset Sample Summary")
        print("------------------------------")
        print("Showing full data attributes and original attributes vs (->) performed transformations\n")
        print(f"{'Total imgs':<12} : {len(self)}")
        print(f"{'Image shape':<12} : {str(self.x.shape[1:]):<20} -> {tuple(sample['pixel_data'].shape)}")
        print(f"{'Image dtype':<12} : {str(self.x.dtype):<20} -> {sample['pixel_data'].dtype}")
        print(f"{'All labels':<12} : {np.unique(self.y)}")
        print(f"{'Label':<12} : {str(self.y.dtype):<20} -> {sample['fire_label']} ({type(sample['fire_label']).__name__})")
        print(f"{'Key':<12} : {str(self.keys.dtype):<20} -> {sample['composite_key']} ({type(sample['composite_key']).__name__})")

In [3]:
test_file = PARAMETERS['PROJ_HOME']/"data"/"Sentinel2"/"2018_B001_20180101_20180117_sentinel_batch.npz"
test_data = SentinelData(test_file)
test_data.sample_summary()

SentinelDataset Sample Summary
------------------------------
Showing full data attributes and original attributes vs (->) performed transformations

Total imgs   : 787
Image shape  : (128, 128, 3)        -> (3, 128, 128)
Image dtype  : float32              -> torch.float32
All labels   : [False  True]
Label        : bool                 -> False (bool_)
Key          : int64                -> 51620180101 (str)


### Class `ResNetFeatureExtractor`

This class encapsulates the feature extraction process - Turns the Sentinel2 images into 512 feature vector

In [8]:
class ResNetFeatExtractor(nn.Module):
    def __init__(self):
        # Initialize nn.Module class before defining FeatureExtractor
        super().__init__()

        # Load pre trained weights
        weights = ResNet18_Weights.DEFAULT
        self.model = resnet18(weights = weights)
        # Remove final classification layer (as we only need feature extraction)
        self.extractor = create_feature_extractor(self.model, 
                                                  return_nodes = {"avgpool": "features"})
        # Freeze weights
        for param in self.model.parameters():
            param.requires_grad = False

    def forward(self, x):
        features = self.extractor(x)
        return features['features'].flatten(1)

In [9]:

loader = DataLoader(test_data, batch_size = 32, shuffle = False)
batch = next(iter(loader))
print(batch['pixel_data'].shape) 
print(batch["fire_label"][:5])
print(batch["composite_key"][:5])


torch.Size([32, 3, 128, 128])
tensor([False, False, False, False, False])
['51620180101', '197120180101', '46420180101', '27020180101', '152420180101']


In [10]:
model = ResNetFeatExtractor()
img = batch['pixel_data']
with torch.no_grad():
    features = model(img)

print(features.shape)

torch.Size([32, 512])


In [11]:
print(features)

tensor([[0.2250, 1.0879, 0.7368,  ..., 0.2686, 0.6329, 0.0000],
        [0.4735, 0.1369, 2.0451,  ..., 1.0127, 1.3240, 0.2196],
        [0.4397, 1.1269, 1.9944,  ..., 1.3039, 0.1850, 3.5827],
        ...,
        [1.5455, 1.0852, 0.9104,  ..., 0.3212, 1.4653, 0.0407],
        [0.4318, 1.2318, 2.9762,  ..., 0.4681, 0.9135, 0.9103],
        [0.0921, 0.3609, 1.1662,  ..., 0.0000, 0.4455, 0.7643]])
